<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/Experiment_7_Prompt_Engineering_Prompting_Strategies_and_Decoding_Parameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAKESH M A  - ENG23CS0157

Experiment 7-Prompt Engineering: Prompting Strategies and Decoding Parameters





In [3]:
!pip install -q transformers accelerate pandas

In [4]:
import re
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

DEV = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", DEV)

if DEV == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
Device: cuda
GPU: Tesla T4


In [5]:
tok = AutoTokenizer.from_pretrained(
    MODEL,
    padding_side="left"
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if DEV == "cuda" else torch.float32
).to(DEV).eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
@torch.no_grad()
def generate(
    prompt,
    system="You are a helpful, concise assistant.",
    temperature=0.0,
    top_k=50,
    top_p=1.0,
    max_new_tokens=200
):

    text = tok.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    enc = tok(text, return_tensors="pt").to(DEV)

    kw = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tok.pad_token_id
    )

    if temperature > 0:
        kw.update(
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )
    else:
        kw.update(do_sample=False)

    out = model.generate(**enc, **kw)

    return tok.decode(
        out[0, enc["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

In [7]:
DATA = [
    ("A shop sells pens at 12 rupees each. Riya buys 7 pens. How much does she pay?", "84"),

    ("A train travels 60 km/h for 3 hours, then 40 km/h for 2 hours. Total distance in km?", "260"),

    ("Tom has 45 marbles and gives away one third of them. How many are left?", "30"),

    ("A class has 32 students; 3/8 are girls. How many boys are there?", "20"),

    ("Anu reads 15 pages a day. How many days to finish a 240-page book?", "16"),

    ("A bag has 5 red and 7 blue balls. 4 blue balls are removed. How many balls remain?", "8"),
]

print("Total questions:", len(DATA))

Total questions: 6


In [8]:
FEW_SHOT = (
    "Q: Sam has 3 boxes with 4 apples each. How many apples?\n"
    "A: 12\n\n"

    "Q: A pencil costs 5 rupees. How much do 6 pencils cost?\n"
    "A: 30\n\n"
)

COT_SHOT = (
    "Q: Sam has 3 boxes with 4 apples each. How many apples?\n"
    "A: Each box has 4 apples and there are 3 boxes, so 3 x 4 = 12. The answer is 12.\n\n"

    "Q: A pencil costs 5 rupees. How much do 6 pencils cost?\n"
    "A: One pencil is 5 rupees, so 6 pencils cost 6 x 5 = 30. The answer is 30.\n\n"
)

STRATEGIES = {

    "zero-shot": lambda q:
        f"{q}\nGive only the final number.",

    "few-shot": lambda q:
        f"{FEW_SHOT}Q: {q}\nA:",

    "zero-shot-CoT": lambda q:
        f"{q}\nLet's think step by step, then end with 'The answer is <number>'.",

    "few-shot-CoT": lambda q:
        f"{COT_SHOT}Q: {q}\nA:"
}

print("Prompting strategies defined:")
for strategy in STRATEGIES:
    print(strategy)

Prompting strategies defined:
zero-shot
few-shot
zero-shot-CoT
few-shot-CoT


In [9]:
def last_number(text):

    nums = re.findall(
        r"-?\d+(?:\.\d+)?",
        text.replace(",", "")
    )

    return nums[-1] if nums else None


def run(strategy, temperature=0.0, top_k=50, runs=1):

    correct = total = 0
    sample = ""

    for q, gold in DATA:

        for _ in range(runs):

            out = generate(
                STRATEGIES[strategy](q),
                temperature=temperature,
                top_k=top_k
            )

            correct += last_number(out) == gold
            total += 1
            sample = out

    return correct / total, sample


print("Evaluation functions ready!")

Evaluation functions ready!


In [10]:
rows = []

print("== Part A: Prompting Strategies (Greedy) ==")

for s in STRATEGIES:

    acc, sample = run(s)

    rows.append({
        "experiment": "strategy",
        "setting": s,
        "accuracy": round(acc, 3)
    })

    print(
        f"{s:15s} accuracy={acc:.2f} | "
        f"sample: {sample[:90]!r}"
    )

== Part A: Prompting Strategies (Greedy) ==
zero-shot       accuracy=0.00 | sample: '12'
few-shot        accuracy=0.17 | sample: '8'
zero-shot-CoT   accuracy=0.50 | sample: 'To find out how many balls remain after removing 4 blue balls from the original 12 balls ('
few-shot-CoT    accuracy=0.67 | sample: 'Initially, there are 5 red + 7 blue = 12 balls in the bag. After removing 4 blue balls, th'


In [12]:
print("\n== Part B: Temperature / Top-k ==")

for t, k in [
    (0.0, 50),
    (0.3, 50),
    (0.7, 50),
    (1.0, 50),
    (0.7, 5),
    (0.7, 100)
]:

    acc, _ = run(
        "few-shot-CoT",
        temperature=t,
        top_k=k,
        runs=3 if t > 0 else 1
    )

    rows.append({
        "experiment": "decoding",
        "setting": f"T={t}, top_k={k}",
        "accuracy": round(acc, 3)
    })

    print(
        f"temperature={t} "
        f"top_k={k} "
        f"accuracy={acc:.2f}"
    )


== Part B: Temperature / Top-k ==
temperature=0.0 top_k=50 accuracy=0.67
temperature=0.3 top_k=50 accuracy=0.72
temperature=0.7 top_k=50 accuracy=0.67
temperature=1.0 top_k=50 accuracy=0.61
temperature=0.7 top_k=5 accuracy=0.56
temperature=0.7 top_k=100 accuracy=0.44


In [13]:
results = pd.DataFrame(rows)

print("\nFINAL EXPERIMENT RESULTS")

display(results)

results.to_csv("exp7_results.csv", index=False)

print("\nResults saved successfully!")


FINAL EXPERIMENT RESULTS


,experiment,setting,accuracy
0,strategy,zero-shot,0.000
1,strategy,few-shot,0.167
2,strategy,zero-shot-CoT,0.500
3,strategy,few-shot-CoT,0.667
4,decoding,"T=0.0, top_k=50",0.667
5,decoding,"T=0.0, top_k=50",0.667
6,decoding,"T=0.3, top_k=50",0.722
7,decoding,"T=0.7, top_k=50",0.667
8,decoding,"T=1.0, top_k=50",0.611
9,decoding,"T=0.7, top_k=5",0.556



Results saved successfully!


In [14]:
from google.colab import files

files.download("exp7_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>